# ============================================================
# Chapter 1: Efficient Inference in Large Language Models
# ============================================================

# Setup - Import Libraries and Load GPT-2 Model

In [1]:
import torch  # Main PyTorch library for tensor operations
import torch.nn as nn  # Neural network modules (not directly used in this example)
import time  # Used for measuring execution time (not used in this example)
!pip install transformers

from transformers import GPT2LMHeadModel, GPT2Tokenizer  # GPT-2 model and tokenizer classes

print("Setting up models...")  # Display setup status

# Load the GPT-2 tokenizer
# The tokenizer converts text into token IDs and vice versa
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Load the pre-trained GPT-2 language model
# GPT2LMHeadModel is GPT-2 with a language modeling head for next-token prediction
model = GPT2LMHeadModel.from_pretrained("gpt2")

print("Setup complete.")  # Confirm successful loading


# ============================================================
# Topic: Autoregressive Text Generation with GPT-2
# Purpose:
# - Demonstrate how GPT-2 generates text one token at a time
# - Show the autoregressive process used by modern LLMs
# - Use greedy decoding (always choose the most probable token)
# ============================================================

# Starting text prompt given to the model
prompt = "The next day is bright"

# Convert the text prompt into token IDs
# return_tensors="pt" returns PyTorch tensors
inputs = tokenizer(prompt, return_tensors="pt")

# Extract token IDs from the tokenizer output
# Shape: (1, sequence_length)
input_ids = inputs.input_ids

# Print the initial prompt
print(f"Prompt: '{prompt}'", end="")

# Generate 20 new tokens one by one
for _ in range(20):

    # --------------------------------------------------------
    # Step 1: Pass the entire current sequence to GPT-2
    # --------------------------------------------------------

    # GPT-2 predicts the next token for every position
    outputs = model(input_ids)

    # Logits are raw prediction scores before softmax
    # Shape: (batch_size, sequence_length, vocab_size)
    logits = outputs.logits

    # --------------------------------------------------------
    # Step 2: Get prediction for the last position only
    # --------------------------------------------------------

    # We only care about the prediction after the final token
    # ":" keeps all batches
    # "-1" selects the last token position
    # ":" keeps all vocabulary scores
    #
    # Shape: (1, vocab_size)
    next_token_logits = logits[:, -1, :]

    # --------------------------------------------------------
    # Step 3: Choose the most likely next token
    # --------------------------------------------------------

    # Greedy decoding:
    # Select the token with the highest logit score
    #
    # Shape after argmax: (1,)
    next_token_id = torch.argmax(next_token_logits, dim=-1)

    # Add an extra dimension so it can be concatenated
    #
    # Shape becomes: (1, 1)
    next_token_id = next_token_id.unsqueeze(-1)

    # --------------------------------------------------------
    # Step 4: Append the predicted token to the sequence
    # --------------------------------------------------------

    # Concatenate the new token to the existing sequence
    #
    # Before:
    # [The, next, day, is, bright]
    #
    # After:
    # [The, next, day, is, bright, predicted_token]
    #
    # Shape increases by one token each iteration
    input_ids = torch.cat([input_ids, next_token_id], dim=-1)

    # --------------------------------------------------------
    # Step 5: Convert token ID back to readable text
    # --------------------------------------------------------

    # Decode the generated token into text
    new_token = tokenizer.decode(next_token_id[0])

    # Print the token immediately
    # flush=True forces instant display
    print(new_token, end="", flush=True)

# Print a newline after generation is complete
print("\n")


# ============================================================
# What Happens Internally?
# ============================================================
#
# Prompt:
# "The next day is bright"
#
# Step 1:
# GPT-2 reads:
# "The next day is bright"
#
# Predicts:
# " and"
#
# Step 2:
# GPT-2 now reads:
# "The next day is bright and"
#
# Predicts:
# " sunny"
#
# Step 3:
# GPT-2 now reads:
# "The next day is bright and sunny"
#
# Predicts another token
#
# This process repeats until 20 tokens are generated.
#
# This is called AUTOREGRESSIVE GENERATION because each newly
# generated token becomes part of the input for generating
# the next token.
#
# Input Shape Progression:
#
# Initial:
# (1, 5)
#
# After 1 token:
# (1, 6)
#
# After 2 tokens:
# (1, 7)
#
# ...
#
# After 20 generated tokens:
# (1, 25)
#
# ============================================================

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/dill-0.3.9-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/looseversion-1.3.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/opt_einsum-3.4.0-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/nvfuser-0.2.26a0+c5e1555-py3.12-linux-x86_64.egg is deprecated. pip 25.1 wi

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting up models...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11173.94it/s]


Setup complete.
 and sunny, and the sun is shining. The sun is shining, and the moon is shining.



# Measuring the Speed Improvement of KV Cache

In [2]:
# Starting prompt for generation
prompt = "The next day is bright"

# Convert text into token IDs and attention mask
# return_tensors="pt" returns PyTorch tensors
inputs = tokenizer(prompt, return_tensors="pt")

# Token IDs representing the input text
# Shape: (batch_size, sequence_length)
input_ids = inputs.input_ids

# Attention mask tells the model which tokens are real
# and which are padding tokens
# Shape: (batch_size, sequence_length)
attention_mask = inputs.attention_mask


# ============================================================
# Part 1: Measure Generation Time WITHOUT KV Cache
# ============================================================

print("Generating without KV Cache...")

# Record starting time
start_time_without_cache = time.time()

# Generate 100 new tokens
output_without_cache = model.generate(
    
    # Input token sequence
    input_ids,

    # Generate 100 additional tokens
    max_new_tokens=100,

    # Disable KV Cache
    # Model recomputes attention for ALL previous tokens
    # at every generation step
    use_cache=False,

    # Attention mask for valid tokens
    attention_mask=attention_mask
)

# Record ending time
end_time_without_cache = time.time()

# Total execution time
duration_without_cache = end_time_without_cache - start_time_without_cache

# Display runtime
print(f"Time without KV Cache: {duration_without_cache:.4f} seconds\n")


# ============================================================
# Part 2: Measure Generation Time WITH KV Cache
# ============================================================

print("Generating with KV Cache...")

# Record starting time
start_time_with_cache = time.time()

# Generate 100 new tokens
output_with_cache = model.generate(

    # Same input sequence
    input_ids,

    # Generate 100 new tokens
    max_new_tokens=100,

    # Enable KV Cache
    # Previously computed Keys and Values are stored
    # and reused instead of being recomputed
    use_cache=True,

    # Attention mask
    attention_mask=attention_mask
)

# Record ending time
end_time_with_cache = time.time()

# Total execution time
duration_with_cache = end_time_with_cache - start_time_with_cache

# Display runtime
print(f"Time with KV Cache: {duration_with_cache:.4f} seconds\n")


# ============================================================
# Part 3: Calculate Speedup
# ============================================================

# Speedup formula
#
# Example:
# Without Cache = 10 seconds
# With Cache    = 2 seconds
#
# Speedup = 10 / 2 = 5x
speedup = duration_without_cache / duration_with_cache

# Display speedup factor
print(f"KV Cache Speedup: {speedup:.2f}x")


# ============================================================
# Deep Explanation: Why KV Cache is Faster
# ============================================================
#
# Assume prompt:
#
# "The next day is bright"
#
# Tokenized as:
#
# [The] [next] [day] [is] [bright]
#
#
# ------------------------------------------------------------
# WITHOUT KV CACHE
# ------------------------------------------------------------
#
# Step 1:
# Model processes:
#
# [The] [next] [day] [is] [bright]
#
# Generates:
#
# [sunny]
#
#
# Step 2:
# Model processes EVERYTHING AGAIN:
#
# [The] [next] [day] [is] [bright] [sunny]
#
#
# Step 3:
# Model processes EVERYTHING AGAIN:
#
# [The] [next] [day] [is] [bright] [sunny] [and]
#
#
# Step 4:
# Model processes EVERYTHING AGAIN:
#
# [The] [next] [day] [is] [bright] [sunny] [and] [warm]
#
#
# Repeated computation causes increasing cost.
#
#
# ------------------------------------------------------------
# WITH KV CACHE
# ------------------------------------------------------------
#
# Step 1:
# Compute attention for:
#
# [The] [next] [day] [is] [bright]
#
# Store:
#   K = Key vectors
#   V = Value vectors
#
#
# Step 2:
# When generating next token:
#
# Reuse stored K and V
#
# Compute attention only for:
#
# [sunny]
#
#
# Step 3:
#
# Reuse previous cache
#
# Compute only for:
#
# [and]
#
#
# Step 4:
#
# Reuse previous cache
#
# Compute only for:
#
# [warm]
#
#
# Huge reduction in computation.
#
#
# ------------------------------------------------------------
# What is Stored?
# ------------------------------------------------------------
#
# For each Transformer layer:
#
# Key Matrix (K)
# Value Matrix (V)
#
# Example:
#
# Layer 1:
#   K1, V1
#
# Layer 2:
#   K2, V2
#
# ...
#
# Layer 12 (GPT-2 Small):
#   K12, V12
#
#
# These tensors are reused during generation.
#
#
# ------------------------------------------------------------
# Why KV Cache is Important in LLMs
# ------------------------------------------------------------
#
# Models such as:
#
# - GPT-2
# - GPT-3
# - GPT-4
# - DeepSeek
# - Llama
# - Claude
# - Gemini
#
# all use KV Cache during inference.
#
# Without KV Cache:
#     Generation becomes increasingly slower as
#     the sequence grows.
#
# With KV Cache:
#     Only the newest token requires computation.
#
# This makes long-text generation practical.
#
#
# ------------------------------------------------------------
# Complexity Comparison
# ------------------------------------------------------------
#
# Without Cache:
#     Recompute entire history every step
#
# Approximate Cost:
#     O(n²)
#
#
# With Cache:
#     Reuse previous Keys and Values
#
# Approximate Cost:
#     O(n)
#
#
# This is why KV Cache is one of the most important
# optimizations used during LLM inference.
# ============================================================

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generating without KV Cache...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Time without KV Cache: 305.6893 seconds

Generating with KV Cache...
Time with KV Cache: 0.7882 seconds

KV Cache Speedup: 387.81x


# Multi-Query Attention (MQA) Implementation

In [3]:
# MHA (Multi-Head Attention):
#     Each head has its own Q, K, V projections
#
# MQA (Multi-Query Attention):
#     Each head has its own Q projection
#     ALL heads share ONE K projection
#     ALL heads share ONE V projection
#
# Benefit:
#     Much smaller KV Cache memory usage
#
# Used in:
#     - PaLM
#     - Gemini
#     - Falcon
#     - Many modern LLMs
# ============================================================

class MultiQueryAttention(nn.Module):

    # --------------------------------------------------------
    # Constructor
    # --------------------------------------------------------
    def __init__(self, d_model, num_heads, dropout=0.0):

        # Initialize parent PyTorch module
        super().__init__()

        # Ensure hidden dimension can be evenly split across heads
        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        # Total embedding dimension
        # Example: 512
        self.d_model = d_model

        # Number of attention heads
        # Example: 8
        self.num_heads = num_heads

        # Dimension per head
        #
        # Example:
        # 512 / 8 = 64
        self.d_head = d_model // num_heads

        # ----------------------------------------------------
        # Query Projection
        # ----------------------------------------------------
        #
        # Creates independent queries for all heads
        #
        # Input:
        #   (B, seq_len, 512)
        #
        # Output:
        #   (B, seq_len, 512)
        self.W_q = nn.Linear(d_model, d_model)

        # ----------------------------------------------------
        # Shared Key Projection
        # ----------------------------------------------------
        #
        # Unlike MHA, only ONE Key projection is created
        #
        # Input:
        #   (B, seq_len, 512)
        #
        # Output:
        #   (B, seq_len, 64)
        self.W_k = nn.Linear(d_model, self.d_head)

        # ----------------------------------------------------
        # Shared Value Projection
        # ----------------------------------------------------
        #
        # Unlike MHA, only ONE Value projection is created
        #
        # Input:
        #   (B, seq_len, 512)
        #
        # Output:
        #   (B, seq_len, 64)
        self.W_v = nn.Linear(d_model, self.d_head)

        # ----------------------------------------------------
        # Output Projection
        # ----------------------------------------------------
        #
        # Combines all head outputs back into d_model dimension
        self.W_o = nn.Linear(d_model, d_model)

        # Dropout layer for regularization
        self.dropout = nn.Dropout(dropout)

        # ----------------------------------------------------
        # Causal Mask
        # ----------------------------------------------------
        #
        # Prevents future token access
        #
        # Example:
        #
        # Token 1 can see:
        # [1]
        #
        # Token 2 can see:
        # [1,2]
        #
        # Token 3 can see:
        # [1,2,3]
        #
        # Upper-triangular matrix of ones
        #
        # Shape:
        # (1,1,1024,1024)
        self.register_buffer(
            'mask',
            torch.triu(
                torch.ones(1, 1, 1024, 1024),
                diagonal=1
            )
        )

    # ========================================================
    # Forward Pass
    # ========================================================
    def forward(self, x):

        # x shape:
        # (batch_size, seq_len, d_model)

        # Extract dimensions
        batch_size, seq_len, _ = x.shape

        # ----------------------------------------------------
        # Step 1: Compute Queries
        # ----------------------------------------------------
        #
        # W_q output:
        # (B, seq_len, d_model)
        q = self.W_q(x)

        # Split into heads
        #
        # Before:
        # (B, seq_len, 512)
        #
        # After:
        # (B, seq_len, 8, 64)
        q = q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.d_head
        )

        # Move heads dimension forward
        #
        # Result:
        # (B, 8, seq_len, 64)
        q = q.transpose(1, 2)

        # ----------------------------------------------------
        # Step 2: Compute Shared Keys
        # ----------------------------------------------------
        #
        # Output:
        # (B, seq_len, 64)
        k = self.W_k(x)

        # Reshape
        #
        # (B, seq_len, 1, 64)
        k = k.view(
            batch_size,
            seq_len,
            1,
            self.d_head
        )

        # Move head dimension
        #
        # (B,1,seq_len,64)
        k = k.transpose(1, 2)

        # ----------------------------------------------------
        # Step 3: Compute Shared Values
        # ----------------------------------------------------
        #
        # Output:
        # (B, seq_len, 64)
        v = self.W_v(x)

        # Reshape
        #
        # (B, seq_len,1,64)
        v = v.view(
            batch_size,
            seq_len,
            1,
            self.d_head
        )

        # Move head dimension
        #
        # (B,1,seq_len,64)
        v = v.transpose(1, 2)

        # ----------------------------------------------------
        # Step 4: Share K and V Across All Heads
        # ----------------------------------------------------
        #
        # Original:
        # (B,1,seq_len,64)
        #
        # Repeat across heads:
        # (B,8,seq_len,64)
        #
        # This is the key idea of MQA.
        #
        # All heads use the same K and V.
        k = k.repeat(1, self.num_heads, 1, 1)

        v = v.repeat(1, self.num_heads, 1, 1)

        # ----------------------------------------------------
        # Step 5: Compute Attention Scores
        # ----------------------------------------------------
        #
        # Formula:
        #
        # Attention(Q,K)
        # = QKᵀ / √d_head
        #
        # Shape:
        # (B, num_heads, seq_len, seq_len)
        attn_scores = (
            q @ k.transpose(-2, -1)
        ) / (self.d_head ** 0.5)

        # ----------------------------------------------------
        # Step 6: Apply Causal Mask
        # ----------------------------------------------------
        #
        # Future positions become -∞
        #
        # Softmax(-∞) = 0
        #
        # Prevents information leakage from future tokens
        attn_scores = attn_scores.masked_fill(
            self.mask[:, :, :seq_len, :seq_len] == 1,
            float('-inf')
        )

        # ----------------------------------------------------
        # Step 7: Convert Scores to Probabilities
        # ----------------------------------------------------
        #
        # Softmax converts raw scores into probabilities
        #
        # Shape remains:
        # (B, num_heads, seq_len, seq_len)
        attn_weights = torch.softmax(
            attn_scores,
            dim=-1
        )

        # Apply dropout
        attn_weights = self.dropout(attn_weights)

        # ----------------------------------------------------
        # Step 8: Compute Context Vectors
        # ----------------------------------------------------
        #
        # Formula:
        #
        # Context = AttentionWeights × V
        #
        # Shape:
        # (B, num_heads, seq_len, d_head)
        context_vector = attn_weights @ v

        # Move heads back
        #
        # (B, seq_len, num_heads, d_head)
        context_vector = context_vector.transpose(1, 2)

        # Ensure contiguous memory layout
        context_vector = context_vector.contiguous()

        # Merge all heads
        #
        # (B, seq_len, 512)
        context_vector = context_vector.view(
            batch_size,
            seq_len,
            self.d_model
        )

        # ----------------------------------------------------
        # Step 9: Final Output Projection
        # ----------------------------------------------------
        #
        # Combines information from all heads
        output = self.W_o(context_vector)

        # Return final attention output
        return output


# ============================================================
# Testing the MQA Layer
# Purpose:
# - Create dummy input
# - Run forward pass
# - Verify output shape
# ============================================================

# Model hidden dimension
d_model = 512

# Number of attention heads
num_heads = 8

# Number of sequences processed together
batch_size = 4

# Number of tokens per sequence
seq_len = 64

# Create MQA layer
mqa_layer = MultiQueryAttention(
    d_model,
    num_heads
)

# Create random input tensor
#
# Shape:
# (4, 64, 512)
dummy_input = torch.randn(
    batch_size,
    seq_len,
    d_model
)

# Run forward pass through MQA
output = mqa_layer(dummy_input)

# Display results
print("MQA Layer successful!")

# Input tensor shape
print(f"Input shape: {dummy_input.shape}")

# Output tensor shape
print(f"Output shape: {output.shape}")


# ============================================================
# MHA vs MQA Memory Example
# ============================================================
#
# Suppose:
#
# d_model = 512
# num_heads = 8
# d_head = 64
#
#
# ------------------------------------------------------------
# Multi-Head Attention (MHA)
# ------------------------------------------------------------
#
# Head 1:
#   K1,V1
#
# Head 2:
#   K2,V2
#
# ...
#
# Head 8:
#   K8,V8
#
# Total:
#   8 Key matrices
#   8 Value matrices
#
#
# ------------------------------------------------------------
# Multi-Query Attention (MQA)
# ------------------------------------------------------------
#
# Head 1:
#   Q1
#
# Head 2:
#   Q2
#
# ...
#
# Head 8:
#   Q8
#
# Shared:
#   K
#   V
#
# Total:
#   1 Key matrix
#   1 Value matrix
#
#
# Result:
#
# KV Cache memory reduced by ~8×
#
# This is why MQA is popular in modern LLM inference.
# ============================================================

MQA Layer successful!
Input shape: torch.Size([4, 64, 512])
Output shape: torch.Size([4, 64, 512])
